In [2]:
import re
import string
from difflib import SequenceMatcher
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score



In [ ]:
# ---- 1. Load pairs ----
pairs = pd.read_csv(r"C:\Users\Praphulla\Downloads\Research\data\processed\abt_buy_pairs.csv")

In [5]:
pairs

,id_abt,id_buy,name_abt,name_buy,label
0,34959,207383660,Plantronics .Audio 920 Bluetooth Headset - AUD...,Plantronics .Audio 920 Wireless Earset - 78592-01,1
1,33804,90125772,Motorola MotoRokr Portable Bluetooth Car Kit S...,Sanus Speaker Mount - WMS3B BLACK,0
2,36243,206678505,Monster iCarPlay Wireless 250 FM Transmitter W...,MONSTER A IP FM-CH 250 iCarPlay Wireless 250 F...,1
3,37010,204559210,Onkyo Black Stereo Receiver - TX8255,Panasonic NNSD797S 1.6 cu. ft. Genius Prestige...,0
4,35477,206359209,Griffin iTrip FM Transmitter - 4052TRPSEB,Griffin iTrip FM Transmitter - 4052-TRPSEB,1
...,...,...,...,...,...
2189,38511,207465595,Audiovox 7' Acrylic Digital Photo Frame - DPF701,Panasonic Lumix DMC-FS3 Digital Camera - Silver,0
2190,31176,205844279,Sony White Cybershot T Series Digital Camera J...,Sony LCJ-THC/B Jacket Case with Stylus - LCJ-T...,1
2191,37856,205520997,Canon Black Leather Case - 3528B001,Bracketron iPod Docking Kit,0
2192,35810,209026638,Canon KP-36IP Color Ink & Paper Set - 7737A001,Toshiba 52RV535U - 52' Widescreen 1080p LCD HD...,0


In [4]:
# ---- 2. Normalization function ----
def normalize(text):
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [6]:
# ---- 3. Similarity functions ----
def jaccard_similarity(name1, name2):
    tokens1 = set(normalize(name1).split())
    tokens2 = set(normalize(name2).split())
    if not tokens1 or not tokens2:
        return 0.0
    intersection = tokens1 & tokens2
    union = tokens1 | tokens2
    return len(intersection) / len(union)

def sequence_similarity(name1, name2):
    return SequenceMatcher(None, normalize(name1), normalize(name2)).ratio()

# Trying Jaccard first (simple, interpretable, close to what you already do with normalization matching)
pairs['pred_score'] = pairs.apply(
    lambda row: jaccard_similarity(row['name_abt'], row['name_buy']), axis=1
)


In [7]:
# ---- 4. Trying multiple thresholds ----
thresholds = [0.2, 0.3, 0.5]

results = []
for t in thresholds:
    preds = (pairs['pred_score'] > t).astype(int)
    p = precision_score(pairs['label'], preds)
    r = recall_score(pairs['label'], preds)
    f1 = f1_score(pairs['label'], preds)
    acc = accuracy_score(pairs['label'], preds)
    results.append({'threshold': t, 'precision': p, 'recall': r, 'f1': f1, 'accuracy': acc})
    print(f"Threshold={t}: precision={p:.3f}, recall={r:.3f}, f1={f1:.3f}, accuracy={acc:.3f}")

results_df = pd.DataFrame(results)
print(results_df)

Threshold=0.2: precision=0.994, recall=0.910, f1=0.950, accuracy=0.952
Threshold=0.3: precision=0.999, recall=0.749, f1=0.856, accuracy=0.874
Threshold=0.5: precision=0.998, recall=0.370, f1=0.540, accuracy=0.685
   threshold  precision    recall        f1  accuracy
0        0.2   0.994024  0.909754  0.950024  0.952142
1        0.3   0.998785  0.749316  0.856250  0.874202
2        0.5   0.997543  0.370100  0.539894  0.684594


In [ ]:
print(results_df)

   threshold  precision    recall        f1  accuracy
0        0.2   0.994024  0.909754  0.950024  0.952142
1        0.3   0.998785  0.749316  0.856250  0.874202
2        0.5   0.997543  0.370100  0.539894  0.684594


In [ ]:
# ---- 5. Save predictions using the best threshold (pick based on best F1) ----
best_threshold = results_df.loc[results_df['f1'].idxmax(), 'threshold']
print(f"\nBest threshold by F1: {best_threshold}")

pairs['pred_label'] = (pairs['pred_score'] > best_threshold).astype(int)

output = pairs[['id_abt', 'id_buy', 'label', 'pred_score', 'pred_label']]
output.to_csv(r"C:\Users\Praphulla\Downloads\Research\data\processed\abt_buy_rulebased_preds.csv", index=False)
print("Saved to data\abt_buy_rulebased_preds.csv")


Best threshold by F1: 0.2
Saved to databt_buy_rulebased_preds.csv
